# Quarb in Jupyter

One query language over every tree — databases, typed notebooks, files, code. Every cell here is a Quarb query; lines starting with `%` are directives. The first cell uses `%python` (the escape hatch) to build two throwaway fixtures and mount them, so the notebook is self-contained.

In [1]:
%python
import sqlite3, tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp())
db = tmp / 'music.db'
con = sqlite3.connect(db)
con.executescript('''
CREATE TABLE artists(id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE albums(id INTEGER PRIMARY KEY, title TEXT,
                    artist_id INTEGER REFERENCES artists(id));
INSERT INTO artists VALUES (1,'Holst'),(2,'Bach');
INSERT INTO albums VALUES (1,'The Planets',1),(2,'Cantatas',2),(3,'Suites',1);
''')
con.commit(); con.close()
lab = tmp / 'lab.kaiv'
lab.write_text('.!kaiv 1\n\n[/@hosts]\nname=web-01\n!float:W\n'
               'draw=142.5\n[]\n\n[/@hosts]\nname=db-01\n!float:W\n'
               'draw=290\n[]\n')
# mount both into the live session (the %python escape reaches it)
cmdb = tmp / 'cmdb.yaml'
cmdb.write_text('artists:\n  - name: Holst\n    country: UK\n  - name: Bach\n    country: DE\n')
session.mount(str(db))
session.mount(str(lab))
# mount the SQLite db and the YAML cmdb TOGETHER, to join across them
session.mount(str(db) + ' ' + str(cmdb))
print('mounted:', sorted(session.docs))

mounted: ['lab', 'music', 'music+cmdb']


## A SQLite database, walked by its foreign keys

A foreign key is an edge; `~>` follows it — no `JOIN`.

In [2]:
/albums/*::artist_id~>::name

Record streams render as tables:

In [3]:
/albums/* | rec("album", ::title, "by", ::artist_id~>::name)

## Typed units from a kaiv notebook

Values carry their units; the criterion converts, not your data. (Switch the default document to the kaiv mount first.)

In [4]:
%use lab
/@hosts/*[::draw > 0.2kW]::name

default: lab


db-01

In [5]:
/@hosts/* | rec("host", /name::, "draw", ::draw)

host,draw
web-01,142.5 W
db-01,290 W


## Join across sources

`%mount a b` mounts several sources under one root, so a single query correlates across them — here the SQLite database `<=>` a YAML file, matched by name.

In [6]:
%use music+cmdb
/music/artists/* <=> /cmdb/artists/*[::name = $*1::name]
  | rec('artist', $*1::name, 'country', ::country)

default: music+cmdb


## Directives

`%docs` lists mounts; `%translate` turns a query you already have into Quarb; `%connect NAME PATH...` attaches to a `qua --resident` daemon for arbors too large to rebuild per notebook. Tab-completion (`/albums/*/`+Tab) offers the arbor's real child names, and `_` in a `%python` cell holds the last result — hand it to pandas with `_.df`.

In [7]:
%use music
%docs

default: music


lab, music, music+cmdb


In [8]:
%translate sql SELECT title FROM albums WHERE artist_id = 1

/albums/*[::artist_id = 1] | rec(::title)
